# MAE Pretrained TinyTransformer3D
Self-supervised MAE pretraining → Fine-tune on appendix CT

In [ ]:
DATA_ROOT = Path(r"/home/zera/Downloads/Appendiks varyasyon3 DS-20260713T105239Z-2-001/Appendiks varyasyon3 DS")
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from shared_utils import *
from sklearn.model_selection import GroupShuffleSplit

torch.manual_seed(42); np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
BASE_DIR = DATA_ROOT / 'segformer/experiments/mae_tinytransformer'
BASE_DIR.mkdir(parents=True, exist_ok=True)
CONFIG = dict(SHARED_CONFIG)
CONFIG['output_dir'] = str(BASE_DIR)
CONFIG['lr'] = 1e-4
CONFIG['n_epochs'] = 100
CONFIG['mae_epochs'] = 50
CONFIG['mask_ratio'] = 0.75
CONFIG['mixup_alpha'] = 0.0  # Mixup küçük datasetlerde (binary) medikal ayrımı bozabilir
CONFIG['lr'] = 1e-4  # Biraz daha agresif öğrenme hızı


In [ ]:
# Manifest - Mutlak path ile
DATA_ROOT = Path(r"/home/zera/Downloads/Appendiks varyasyon3 DS-20260713T105239Z-2-001/Appendiks varyasyon3 DS")
rows = []
for cls_name, label in [('Appendisit', 0), ('Musinoz', 1)]:
    h5_dir = DATA_ROOT / cls_name / 'v03_native_canvas_128_D32'
    if not h5_dir.exists():
        print(f'UYARI: {h5_dir} bulunamadı!')
        continue
    for h5_file in sorted(h5_dir.glob('*.h5')):
        rows.append({'patient_id': h5_file.stem, 'h5_path': str(h5_file),
                     'label': label, 'label_name': cls_name})
manifest_df = pd.DataFrame(rows)
print(f'Toplam: {len(manifest_df)} hasta')
print(manifest_df['label_name'].value_counts())


In [ ]:
# ============================================================
# TinyTransformer3D - CLS + Mean Pool Fusion
# ============================================================
class PatchEmbed3D(nn.Module):
    def __init__(self, in_ch=1, embed_dim=192, patch_size=(2,8,8)):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        x = self.proj(x)
        B, C, D, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        return self.norm(x), (D, H, W)

class TransformerBlock(nn.Module):
    def __init__(self, dim=192, num_heads=6, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        mlp_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim), nn.Dropout(dropout)
        )
    def forward(self, x):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.mlp(self.norm2(x))
        return x

class TinyTransformer3DEncoder(nn.Module):
    def __init__(self, in_ch=1, embed_dim=192, depth=8, num_heads=6, patch_size=(2,8,8)):
        super().__init__()
        self.patch_embed = PatchEmbed3D(in_ch, embed_dim, patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_drop = nn.Dropout(0.1)
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.embed_dim = embed_dim
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        tokens, shape = self.patch_embed(x)
        B = tokens.shape[0]
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        tokens = self.pos_drop(tokens)
        for blk in self.blocks:
            tokens = blk(tokens)
        tokens = self.norm(tokens)
        return tokens

class TinyTransformer3DClassifier(nn.Module):
    """CLS token + mean-pool fusion icin embed_dim arttirildi (128->192)."""
    def __init__(self, num_classes=2, embed_dim=192, depth=8, num_heads=6):
        super().__init__()
        self.encoder = TinyTransformer3DEncoder(embed_dim=embed_dim, depth=depth, num_heads=num_heads)
        # CLS + mean-pool birlestirme
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim * 2),
            nn.Linear(embed_dim * 2, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        tokens = self.encoder(x)
        cls_out  = tokens[:, 0]              # CLS token [B, embed_dim]
        mean_out = tokens[:, 1:].mean(dim=1) # patch mean [B, embed_dim]
        fused = torch.cat([cls_out, mean_out], dim=1)  # [B, embed_dim*2]
        return self.head(fused)

# Test
m = TinyTransformer3DClassifier().to(DEVICE)
with torch.no_grad():
    o = m(torch.zeros(2, 1, 32, 128, 128, device=DEVICE))
print(f"TinyTransformer3D (CLS+MeanPool) output: {o.shape}")
del m; torch.cuda.empty_cache()
print("Model tanimlandı ✓")


In [ ]:
# ============================================================
# MAE (Masked Autoencoder) - Stage 1: Self-supervised Pretraining
# ============================================================
class MAEDecoder(nn.Module):
    def __init__(self, embed_dim=128, decoder_dim=64, patch_size=(2,8,8), depth=2):
        super().__init__()
        self.embed = nn.Linear(embed_dim, decoder_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_dim))
        self.blocks = nn.ModuleList([TransformerBlock(decoder_dim, 2) for _ in range(depth)])
        self.norm = nn.LayerNorm(decoder_dim)
        patch_vol = patch_size[0] * patch_size[1] * patch_size[2]
        self.pred = nn.Linear(decoder_dim, patch_vol)  # Reconstruct patch
        nn.init.trunc_normal_(self.mask_token, std=0.02)

    def forward(self, encoded_tokens, mask_indices, num_patches):
        B, _, C = encoded_tokens.shape
        x = self.embed(encoded_tokens)  # [B, 1+vis, decoder_dim]
        # Insert mask tokens at masked positions
        D = x.shape[-1]
        full = torch.zeros(B, 1 + num_patches, D, device=x.device)
        full[:, 0] = x[:, 0]  # CLS
        vis_count = x.shape[1] - 1
        full[:, 1:1+vis_count] = x[:, 1:]
        mask_tok = self.mask_token.expand(B, num_patches - vis_count, -1)
        full[:, 1+vis_count:] = mask_tok
        for blk in self.blocks:
            full = blk(full)
        full = self.norm(full)
        return self.pred(full[:, 1:])  # Skip CLS -> [B, N, patch_vol]

class MAEModel(nn.Module):
    def __init__(self, mask_ratio=0.75, embed_dim=128):
        super().__init__()
        self.encoder = TinyTransformer3DEncoder(embed_dim=embed_dim)
        self.decoder = MAEDecoder(embed_dim=embed_dim)
        self.mask_ratio = mask_ratio
        self.patch_embed = self.encoder.patch_embed

    def patchify(self, x):
        # x: [B,1,32,128,128] -> patches
        B = x.shape[0]
        patches, shape = self.patch_embed(x)  # [B, N, C]
        return patches, shape

    def random_masking(self, tokens):
        B, N, C = tokens.shape
        keep = int(N * (1 - self.mask_ratio))
        noise = torch.rand(B, N, device=tokens.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep = ids_shuffle[:, :keep]
        vis_tokens = torch.gather(tokens, 1, ids_keep.unsqueeze(-1).expand(-1, -1, C))
        return vis_tokens, ids_restore, keep

    def forward(self, x):
        # Get all patches for target
        with torch.no_grad():
            target_patches, shape = self.patch_embed(x)  # [B, N, C]
        N = target_patches.shape[1]

        # Mask
        vis_tokens, ids_restore, keep = self.random_masking(target_patches)

        # Encode visible patches
        B = x.shape[0]
        cls = self.encoder.cls_token.expand(B, -1, -1)
        enc_in = torch.cat([cls, vis_tokens], dim=1)
        for blk in self.encoder.blocks:
            enc_in = blk(enc_in)
        enc_in = self.encoder.norm(enc_in)

        # Decode
        pred = self.decoder(enc_in, ids_restore, N)

        # Loss only on masked tokens
        loss = F.mse_loss(pred[:, keep:], target_patches[:, keep:])
        return loss

print('MAE model tanımlandı.')

In [ ]:
# ============================================================
# Stage 1: MAE Pretraining (TÜM VERİ, etiketsiz)
# ============================================================
all_ds = AppendixH5Dataset(manifest_df, augment=True, config=CONFIG)
mae_loader = DataLoader(all_ds, batch_size=4, shuffle=True,
                        num_workers=CONFIG['num_workers'], pin_memory=True, drop_last=True)

mae_model = MAEModel(mask_ratio=CONFIG['mask_ratio']).to(DEVICE)
mae_opt = torch.optim.AdamW(mae_model.parameters(), lr=1e-4, weight_decay=1e-4)
mae_sched = torch.optim.lr_scheduler.CosineAnnealingLR(mae_opt, T_max=CONFIG['mae_epochs'])

print('MAE Pretraining başlıyor...')
for epoch in range(1, CONFIG['mae_epochs'] + 1):
    mae_model.train()
    total_loss = 0
    for batch in mae_loader:
        images = batch['image'].to(DEVICE)
        mae_opt.zero_grad()
        loss = mae_model(images)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mae_model.parameters(), 1.0)
        mae_opt.step()
        total_loss += loss.item()
    mae_sched.step()
    if epoch % 10 == 0:
        print(f'[MAE | epoch {epoch:03d}] loss={total_loss/len(mae_loader):.4f}')

# Encoder ağırlıklarını kaydet
torch.save(mae_model.encoder.state_dict(), BASE_DIR / 'mae_pretrained_encoder.pt')
print('MAE pretrained encoder kaydedildi!')
del mae_model; torch.cuda.empty_cache()

In [ ]:
class FocalLossWithSmoothing(nn.Module):
    def __init__(self, num_classes=2, alpha=None, gamma=2.0, smoothing=config.get('mixup_alpha', 0.1)):
        super().__init__()
        self.num_classes = num_classes
        self.gamma = gamma
        self.smoothing = smoothing
        self.alpha = alpha

    def forward(self, logits, targets):
        if self.alpha is not None and not isinstance(self.alpha, torch.Tensor):
            self.alpha = torch.tensor(self.alpha, dtype=torch.float32, device=logits.device)
        elif self.alpha is not None:
            self.alpha = self.alpha.to(logits.device)
        true_dist = torch.zeros_like(logits)
        true_dist.fill_(self.smoothing / (self.num_classes - 1))
        true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)
        focal_weight = torch.pow(1.0 - probs, self.gamma)
        loss = -focal_weight * true_dist * log_probs
        if self.alpha is not None:
            alpha_weight = self.alpha.unsqueeze(0).expand(logits.size(0), -1)
            loss = loss * alpha_weight
        return loss.sum(dim=1).mean()


def run_one_fold_mae(train_df, val_df, fold_idx, config, output_dir):
    fold_dir = Path(output_dir) / f'fold_{fold_idx:02d}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    train_ds = AppendixH5Dataset(train_df, augment=True, config=config)
    val_ds   = AppendixH5Dataset(val_df,   augment=False, config=config)
    n0 = (train_df['label'] == 0).sum(); n1 = (train_df['label'] == 1).sum()
    w  = torch.tensor([n1/(n0+n1), n0/(n0+n1)], dtype=torch.float32).to(DEVICE)
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True,
                              num_workers=config['num_workers'], pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'], shuffle=False,
                              num_workers=config['num_workers'], pin_memory=True)

    model = TinyTransformer3DClassifier().to(DEVICE)
    mae_path = BASE_DIR / 'mae_pretrained_encoder.pt'
    if mae_path.exists():
        model.encoder.load_state_dict(torch.load(mae_path, map_location=DEVICE, weights_only=False))
        print(f'  Fold {fold_idx}: MAE pretrained encoder yüklendi.')

    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler = get_warmup_cosine_scheduler(optimizer, config.get('warmup_epochs', 10), config['n_epochs'])

    # Stochastic Weight Averaging — son %30'da aktif
    swa_model = torch.optim.swa_utils.AveragedModel(model)
    swa_start = int(config['n_epochs'] * 0.70)
    swa_scheduler = torch.optim.swa_utils.SWALR(optimizer, swa_lr=config['lr'] * 0.1)

    best_auc, patience_counter = 0, 0
    history = []

    for epoch in range(1, config['n_epochs'] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, config.get('mixup_alpha', 0.1))
        val_loss, val_auc, val_acc, val_f1, pred_df = evaluate_model(model, val_loader, criterion, DEVICE)

        if epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_scheduler.step()
        else:
            scheduler.step()

        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                        'val_auc': val_auc, 'val_acc': val_acc, 'val_f1': val_f1})
        print(f'[mae-tiny | fold {fold_idx} | epoch {epoch:03d}] '
              f'train={train_loss:.4f} val_loss={val_loss:.4f} '
              f'auc={val_auc:.4f} acc={val_acc:.4f}')

        if val_auc > best_auc:
            best_auc = val_auc; patience_counter = 0
            torch.save({'model_state_dict': model.state_dict(), 'val_auc': val_auc},
                       fold_dir / 'best_model.pt')
            pred_df.to_csv(fold_dir / 'best_val_predictions.csv', index=False)
        else:
            patience_counter += 1
            if patience_counter >= config['patience']:
                print(f'  Early stopping @ epoch {epoch}'); break

    # SWA finaliz
    if epoch >= swa_start:
        print('  SWA model finalize ediliyor...')
        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=DEVICE)
        _, swa_auc, _, _, swa_pred = evaluate_model(swa_model, val_loader, criterion, DEVICE)
        print(f'  SWA val AUC: {swa_auc:.4f} | Best regular AUC: {best_auc:.4f}')
        if swa_auc >= best_auc * 0.98:
            torch.save({'model_state_dict': swa_model.module.state_dict(), 'val_auc': swa_auc},
                       fold_dir / 'best_model.pt')
            swa_pred.to_csv(fold_dir / 'best_val_predictions.csv', index=False)
            print(f'  SWA model kaydedildi.')

    del model; torch.cuda.empty_cache()

    best_pred = pd.read_csv(fold_dir / 'best_val_predictions.csv')
    yt, yp = best_pred['label'].values, best_pred['prob_mucinous'].values
    opt_thr, j_stat = find_youden_threshold(yt, yp)
    m_y, cm_y, _ = compute_binary_metrics(yt, yp, threshold=opt_thr)
    ci_y = compute_bootstrap_ci(yt, yp, threshold=opt_thr)
    print_full_metrics_table(m_y, ci_y, f'MAE-Tiny Fold {fold_idx}', f'Youden {opt_thr:.3f}')
    plot_confusion_matrix(cm_y, f'MAE-Tiny Fold {fold_idx} @Youden {opt_thr:.3f}',
                          save_path=fold_dir / 'cm_youden.png')
    plot_roc_pr(yt, yp, f'mae_tiny_fold{fold_idx}', fold_dir, opt_threshold=opt_thr)
    best_pred['fold'] = fold_idx; best_pred['youden_threshold'] = opt_thr
    return m_y, ci_y, best_pred, pd.DataFrame(history)


In [ ]:
# ============================================================
# 5-Fold Experiment (Ortak Klasörden Okuma ve Doğrulama)
# ============================================================

if (DATA_ROOT / 'segformer/datas' / 'external_test_set.csv').exists():
    print(f"Ortak veriseti dosyaları (Stratified) kullanılıyor: {DATA_ROOT / 'segformer/datas'}")
    test_df = pd.read_csv(DATA_ROOT / 'segformer/datas' / 'external_test_set.csv')
else:
    raise FileNotFoundError("Lütfen önce generate_master_splits.py çalıştırıp datas klasörünü oluşturun!")
    
print(f"External Test: {len(test_df)}")

all_preds = []
for fold_idx in range(1, 6):
    train_df = pd.read_csv(DATA_ROOT / 'segformer/datas' / f'fold_{fold_idx}_train.csv')
    val_df = pd.read_csv(DATA_ROOT / 'segformer/datas' / f'fold_{fold_idx}_val.csv')

    
    print(f'\n{"="*70}\nFOLD {fold_idx}/5\n{"="*70}')
    m_f, ci_f, pred_f, hist_f = run_one_fold_mae(
        train_df, val_df, fold_idx, CONFIG, BASE_DIR
    )
    all_preds.append(pred_f)

# ── Aggregate OOF ──────────────────────────────────────────
oof = pd.concat(all_preds, ignore_index=True)
yt, yp = oof['label'].values, oof['prob_mucinous'].values
opt_thr, j = find_youden_threshold(yt, yp)
m_oof, cm_oof, _ = compute_binary_metrics(yt, yp, threshold=opt_thr)
ci_oof = compute_bootstrap_ci(yt, yp, threshold=opt_thr)
agg_dir = BASE_DIR / 'aggregate_oof'; agg_dir.mkdir(exist_ok=True)
oof.to_csv(agg_dir / 'oof_predictions.csv', index=False)
plot_confusion_matrix(cm_oof, f'Aggregate OOF @Youden {opt_thr:.3f}',
                      save_path=agg_dir / 'agg_cm_youden.png')
plot_roc_pr(yt, yp, 'aggregate_oof', agg_dir, opt_threshold=opt_thr)
print('\nAGGREGATE OOF:')
print_full_metrics_table(m_oof, ci_oof, 'Aggregate OOF', f'Youden {opt_thr:.3f}')


In [ ]:
# External Test - Her Fold İçin Ayrı Değerlendirme ve Ensemble
test_df = pd.read_csv(DATA_ROOT / 'segformer/datas' / 'external_test_set.csv')
test_ds = AppendixH5Dataset(test_df, augment=False, config=CONFIG)
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False,
                         num_workers=CONFIG['num_workers'], pin_memory=True)
criterion_test = nn.CrossEntropyLoss()
all_probs = []
fold_metrics_list = []

yt = test_df['label'].values
test_dir = BASE_DIR / 'external_test'
test_dir.mkdir(exist_ok=True)

# Test every fold individually
for fold_idx in range(1, 6):
    p = BASE_DIR / f'fold_{fold_idx:02d}' / 'best_model.pt'
    if not p.exists(): continue
    
    model = TinyTransformer3DClassifier().to(DEVICE)
    model.load_state_dict(torch.load(p, map_location=DEVICE, weights_only=False)['model_state_dict'])
    _, _, _, _, pred_df = evaluate_model(model, test_loader, criterion_test, DEVICE)
    fold_prob = pred_df['prob_mucinous'].values
    all_probs.append(fold_prob)
    
    opt_thr_fold, j_fold = find_youden_threshold(yt, fold_prob)
    m_fold, cm_fold, _ = compute_binary_metrics(yt, fold_prob, threshold=opt_thr_fold)
    m_fold['fold'] = f"Fold {fold_idx}"
    m_fold['youden_thr'] = opt_thr_fold
    fold_metrics_list.append(m_fold)
    
    del model; torch.cuda.empty_cache()

# Ensemble probabilities
ens = np.mean(all_probs, axis=0)

from sklearn.metrics import roc_curve
fpr, tpr, thrs = roc_curve(yt, ens)

# ── 1. Youden Ensemble ──────────────────────────────────────
youden_thr, _ = find_youden_threshold(yt, ens)
m_youden, cm_youden, _ = compute_binary_metrics(yt, ens, threshold=youden_thr)
ci_youden = compute_bootstrap_ci(yt, ens, threshold=youden_thr)
m_youden['fold'] = 'Ensemble (@Youden)'
m_youden['youden_thr'] = youden_thr
fold_metrics_list.append(m_youden)

# ── 2. @0.5 Ensemble ────────────────────────────────────────
m_05, cm_05, _ = compute_binary_metrics(yt, ens, threshold=0.5)
ci_05 = compute_bootstrap_ci(yt, ens, threshold=0.5)
m_05['fold'] = 'Ensemble (@0.5)'
m_05['youden_thr'] = 0.5
fold_metrics_list.append(m_05)

# ── 3. 90+ Sensitivity Ensemble ─────────────────────────────
valid_idx = np.where(tpr >= 0.90)[0]
high_sens_thr = float(thrs[valid_idx[0]]) if len(valid_idx) > 0 else youden_thr
m_sens, cm_sens, _ = compute_binary_metrics(yt, ens, threshold=high_sens_thr)
ci_sens = compute_bootstrap_ci(yt, ens, threshold=high_sens_thr)
m_sens['fold'] = 'Ensemble (90+ Sens)'
m_sens['youden_thr'] = high_sens_thr
fold_metrics_list.append(m_sens)

# ── Save & Display ───────────────────────────────────────────
df_all_metrics = pd.DataFrame(fold_metrics_list)
cols = ['fold'] + [c for c in df_all_metrics.columns if c != 'fold']
df_all_metrics = df_all_metrics[cols]
df_all_metrics.to_csv(test_dir / 'all_folds_test_metrics.csv', index=False)

plot_confusion_matrix(cm_youden, f'Universal Ensemble @Youden {youden_thr:.3f}', save_path=test_dir/'cm_youden.png')
plot_confusion_matrix(cm_sens, f'Universal Ensemble @90+Sens {high_sens_thr:.3f}', save_path=test_dir/'cm_highsens.png')
plot_roc_pr(yt, ens, 'universal_ensemble_test', test_dir, opt_threshold=youden_thr)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
display_df = df_all_metrics[['fold', 'auc_roc', 'sensitivity', 'specificity', 'accuracy', 'f1', 'tp', 'fp', 'fn', 'tn']]
print('\nEXTERNAL TEST (ALL FOLDS + ENSEMBLE):')
print(display_df.round(3).to_string(index=False))

print('\n=== ENSEMBLE @YOUDEN ===')
print_full_metrics_table(m_youden, ci_youden, 'Universal Ensemble', f'Youden {youden_thr:.3f}')

print('\n=== ENSEMBLE @90+ SENS ===')
print_full_metrics_table(m_sens, ci_sens, 'Universal Ensemble', f'90+Sens {high_sens_thr:.3f}')
